# Project 03: Customer Churn / Sales Trend Analysis

**Business question:** Which months and product categories generate the most revenue, how concentrated is revenue among high-value customers, and what practical actions could a manager test?

**Data note:** This notebook creates 2,400 synthetic practice orders from 300 fictional customer IDs for 2025. No real customer data is used. Revenue is in arbitrary synthetic units. A fixed random seed makes results reproducible.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

## 1. Create and inspect the dataset

In [ ]:
rng = np.random.default_rng(2026)
n_orders, n_customers = 2400, 300
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
ids = np.arange(1001, 1001 + n_customers)
weights = rng.lognormal(0, 0.65, n_customers)
weights /= weights.sum()
order_dates = pd.to_datetime(rng.choice(dates, n_orders))
customer_ids = rng.choice(ids, n_orders, p=weights)
categories = ["Electronics", "Fashion", "Home", "Books"]
picked_categories = rng.choice(categories, n_orders, p=[.27, .36, .22, .15])
m = order_dates.month
season = np.select([m == 11, m == 12, np.isin(m, [5, 6]), np.isin(m, [1, 2])],
                   [1.45, 1.30, 1.12, .92], default=1.0)
base = pd.Series(picked_categories).map(
    {"Electronics": 2600, "Fashion": 950, "Home": 1700, "Books": 450}
).to_numpy()
amount = np.round(rng.gamma(2.1, base / 2.1) * season, 2)
orders = pd.DataFrame({
    "order_id": [f"ORD{i:05d}" for i in range(1, n_orders + 1)],
    "order_date": order_dates, "customer_id": customer_ids,
    "category": picked_categories, "amount": amount
}).sort_values("order_date").reset_index(drop=True)

display(orders.head())
print("Rows and columns:", orders.shape)
print("Missing values:\\n", orders.isna().sum())
print("Duplicate order IDs:", orders.order_id.duplicated().sum())

## 2. Feature engineering: extract date information

In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders.order_date.dt.to_period("M").astype(str)
orders["month_name"] = orders.order_date.dt.strftime("%b")
orders["weekday"] = orders.order_date.dt.day_name()
orders["quarter"] = orders.order_date.dt.to_period("Q").astype(str)
display(orders[["order_date", "month", "month_name", "weekday", "quarter"]].head())

## 3. Monthly revenue and month-over-month growth

In [ ]:
monthly = orders.groupby("month").agg(
    revenue=("amount", "sum"),
    orders=("order_id", "count")
)
monthly["mom_growth_pct"] = monthly.revenue.pct_change() * 100
display(monthly.round(2))

best_month = monthly.revenue.idxmax()
worst_month = monthly.revenue.idxmin()
print("Highest revenue month:", best_month)
print("Lowest revenue month:", worst_month)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
monthly.revenue.plot(marker="o", ax=ax)
ax.set_title("Monthly e-commerce revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (synthetic units)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Category comparison and pivot table

In [ ]:
category_summary = orders.groupby("category").amount.agg(
    total_revenue="sum",
    average_order_value="mean",
    order_count="count"
).sort_values("total_revenue", ascending=False)
category_summary["revenue_share_pct"] = (
    category_summary.total_revenue / orders.amount.sum() * 100
)
display(category_summary.round(2))

In [ ]:
monthly_category_pivot = pd.pivot_table(
    orders, values="amount", index="month",
    columns="category", aggfunc="sum", fill_value=0
)
display(monthly_category_pivot.round(2))

In [ ]:
category_summary.total_revenue.sort_values().plot(
    kind="barh", figsize=(8, 5), title="Revenue by product category"
)
plt.xlabel("Revenue (synthetic units)")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

## 5. Top customers: top 10% revenue contribution

In [ ]:
customer_spend = orders.groupby("customer_id").amount.sum().sort_values(ascending=False)
top_n = math.ceil(len(customer_spend) * 0.10)
top_share = customer_spend.head(top_n).sum() / orders.amount.sum() * 100

print("Unique customers:", len(customer_spend))
print("Top 10% customer count:", top_n)
print(f"Top 10% revenue contribution: {top_share:.1f}%")
top_10_customers = customer_spend.head(10)
display(top_10_customers.rename("total_spend").to_frame())

In [ ]:
top_10_customers.sort_values().plot(
    kind="barh", figsize=(8, 5), title="Top 10 customers by total spend"
)
plt.xlabel("Customer spend (synthetic units)")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.show()

## 6. Business recommendations

1. **Peak-month planning:** Prepare stock and campaign capacity before the month with the highest observed revenue. Check multiple years before assuming this is a stable seasonal pattern.
2. **Category focus:** Review availability and marketing for the category with the largest revenue share. Check profit margin and return rates before increasing investment.
3. **Customer retention:** Test a modest loyalty offer for the top 10% customer group. Compare repeat purchases and incremental revenue with a control group.

Revenue is not profit. This dataset does not include costs, returns, discounts, or marketing spend.

## 7. Export results and final business figures

In [ ]:
from pathlib import Path
output = Path("project_output")
(output / "charts").mkdir(parents=True, exist_ok=True)
(output / "data").mkdir(exist_ok=True)

orders.to_csv(output / "data" / "ecommerce_orders_synthetic.csv", index=False)
monthly.to_csv(output / "monthly_revenue_summary.csv")
category_summary.to_csv(output / "category_summary.csv")
monthly_category_pivot.to_csv(output / "monthly_category_pivot.csv")
top_10_customers.rename("total_spend").to_csv(output / "top_10_customers.csv")

print(f"Total revenue: {orders.amount.sum():,.2f} synthetic units")
print(f"Average order value: {orders.amount.mean():,.2f} synthetic units")
print("Best month:", monthly.revenue.idxmax())
print("Worst month:", monthly.revenue.idxmin())
print("Best category:", category_summary.index[0])
print(f"Top 10% customer revenue share: {top_share:.1f}%")
print("Data type: generated synthetic practice data.")

## Conclusion

This project uses date features, monthly trend calculations, category `groupby`, a pivot table, and customer-level spending analysis to find patterns in synthetic practice data. The results support three actions to test: prepare for the peak month, monitor the leading category, and test retention offers for high-value customers.

**Limitation:** The dataset is synthetic and covers one year. It demonstrates an analysis workflow but does not establish real customer behaviour or actual business performance.